# 第 11 章　面向对象编程（上）

> **本章定位**：把"数据"和"操作数据的方法"打包成一个整体。从本章起，程序的主角由"函数"变成"对象"。
>
> **知识点范围**：类的定义 `class`、实例化对象、`__init__` 构造方法、`self`、实例属性、类属性、实例方法、私有属性的初步概念（`__name` 名称改写）、`__str__` 与 `info()` 式自描述方法。
>
> **本章约定**：可用前 10 章全部内容；**不涉及继承与多态**（第 12 章）。

> **先修知识**：第 01—10 章（函数、模块导入、字典）。类中的方法在写法上与函数一致，只是第一个参数固定为 `self`。
>
> **学习目标**：完成本章后，你能够：
>
> 1. 定义含 `__init__` 与 3 个实例属性的类，实例化 2 个对象并打印各自的属性值；
> 2. 分别修改类属性与实例属性各 1 次，运行后说明另一个对象是否受影响；
> 3. 用 `self` 编写 2 个实例方法，一个修改属性、一个只读属性，调用后打印状态变化；
> 4. 用名称改写定义 1 个私有属性，写出从类外访问时的现象与正确读取方式；
> 5. 为类实现 `__str__` 与 `info()`，使 `print(对象)` 与 `对象.info()` 的输出一致。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 11.1 — 11.5 | 类、属性、方法、封装 + 随堂示例 |
| 第二部分 | 代码示范 | 案例 11-1、案例 11-2 | 车辆类 / 信号灯类 |
| 第三部分 | 布置练习 | 练习 11-1、练习 11-2 | 停车场类 / 路口类 |
| 第四部分 | 习题答案 | 答案 11-1、答案 11-2 | 对照解析与评分要点 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 11.1 | 类与对象、实例化 | 11.2 | 案例 11-1、案例 11-2 | 练习 11-1、练习 11-2 |
| 11.2 | `__init__` 与 `self` | 11.3 | 案例 11-1、案例 11-2 | 两个练习 |
| 11.3 | 实例属性与类属性 | 11.4 | 案例 11-1（`total`）、案例 11-2（`PHASES`） | 练习 11-2 |
| 11.4 | 实例方法与自描述（`info` / `__str__`） | 11.5 | 两个案例 | 两个练习 |
| 11.5 | 私有属性与受控访问 | 11.6 | 案例 11-2 | 练习 11-1 |

## 教学提示（教师用）

- **建议课时**：8 学时（4 次课），对应课程标准模块 11。
- **讲解主线**：类是"数据 + 行为"的模板，对象是按模板造出来的实例。沿 `class` → `__init__` → `self` → 实例属性 / 类属性 → 方法 → 私有属性 的顺序推进，每一步都用交通对象落地。
- **学生常见卡点**：方法定义忘记写 `self`；类属性与实例属性混用（`Vehicle.total` 与 `v.total` 含义不同）；`__str__` 与 `__repr__` 混淆；误以为 `__x` 是"绝对私有"（实际只是改名，仍可用 `_类名__x` 访问）。
- **课堂活动建议**：把第 5 章的字典台账重写为 `Vehicle` 对象列表，对比 `d["speed"]` 与 `v.speed`，并让学生说出"哪个更不容易写错键名"。

---

# 第一部分　知识讲解

## 11.1 场景导入：字典能存数据，但不能带行为

用字典描述一辆车是 `{"plate": "京A12345", "speed": 62}`，能存数据但**不能带行为**：想让车"加速 10 km/h"，只能另写一个函数接收字典：

```text
accelerate(car_dict, 10)      # 面向过程：数据与操作分离
car.accelerate(10)            # 面向对象：数据与操作绑定
```

后者更贴近真实世界的表达，也带来三个实际好处：

1. **内聚**：车速的上限 120 写在类里，所有修改车速的路径都经过同一个方法；
2. **安全**：私有属性 + 校验方法，防止外部把绿灯改成 0 秒；
3. **可扩展**：新增"货车"只需新增子类（第 12 章），不必改调用方。

## 11.2 知识点 11.1　类与对象、实例化

### 11.2.1 核心概念

- **类**（class）是模板，描述"这类事物有哪些属性、能做哪些事"；
- **对象**（实例）是按模板造出来的具体个体；
- **实例化**即"按类创建对象"：`v1 = Vehicle("京A12345", "小型车")`。

一个类可创建任意多个对象，每个对象各自持有一份属性值，互不影响。

**原理补充（概念深化）**：`class` 语句执行时会在内存里创建一个类对象，它本身也是对象；而 `类名()` 才是实例化：解释器先分配一块空实例，再调用 `__init__` 初始化，最后返回其引用。每个实例默认携带自己的 `__dict__` 属性字典，这正是各对象属性互不影响的内存依据——`v1.plate = ...` 只是往 `v1.__dict__` 里塞一个键，丝毫不会动 `v2.__dict__`。`is` 比较的是对象身份（可理解为内存中的同一块），`==` 比较值；`type(obj).__name__` 读的是类对象的名字而非实例数据。常见误解是把 `v1 = Vehicle`（没写括号）当成实例化，那其实只是给类对象又绑了一个名字，后续访问实例属性必然 `AttributeError`。另外，类体在定义时就会执行一次，所以类属性此刻已建立，而实例属性要等 `__init__` 被调用才出现。

### 11.2.2 语法要点

| 要素 | 写法 | 说明 |
|---|---|---|
| 定义类 | `class Vehicle:` | 类名用**大驼峰**（`ParkingLot`、`TrafficLight`） |
| 类体 | 缩进 4 空格 | 可含属性、方法、文档字符串 |
| 实例化 | `v1 = Vehicle("京A12345", "小型车")` | 参数自动传给 `__init__` |
| 访问属性 | `v1.speed` | 点号访问 |
| 调用方法 | `v1.accelerate(10)` | 括号内是"除 `self` 之外"的参数 |
| 判断类型 | `type(v1)` / `isinstance(v1, Vehicle)` | 后者第 12 章详述 |

### 11.2.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 类名用小写或下划线 | 不符合规范 | `class vehicle:` 能跑，但应 `Vehicle` |
| 忘记括号 | `v1 = Vehicle` 得到类本身，不是对象 | 实例化必须写 `()` |
| 参数个数不符 | `TypeError: __init__() missing 1 required positional argument` | 对照 `__init__` 签名 |
| 类定义后未实例化就访问属性 | `AttributeError` | 先创建对象 |

### 11.2.4 随堂示例：先造对象，再挂属性

**核心要点**：验证「类是模板、对象是实例」：`Vehicle()` 每次实例化都产生独立对象，给其中一个挂属性不会影响另一个。

In [ ]:
class Vehicle:
    """车辆类：暂时只是个空模板"""

v1 = Vehicle()
v1.plate = "京A12345"
v2 = Vehicle()
v2.plate = "京B67890"

print("v1：", v1.plate)
print("v2：", v2.plate)
print("类型：", type(v1).__name__)
print("两个是不同对象：", v1 is not v2)

**预期输出**：

```text
v1： 京A12345
v2： 京B67890
类型： Vehicle
两个是不同对象： True
```

**变式代码（迁移练习）**：把 demo11-1 的打印 `plate` 值改为打印两个实例的 `__dict__` 与 `is` 身份，验证每个实例各持一份属性字典、动态挂属性互不干扰。

In [ ]:
# 类是模板：两个卡口对象各有独立属性字典，给 g1 挂属性不影响 g2
class Gate:
    pass
g1 = Gate()
g2 = Gate()
g1.plate = "京A12345"
print("g1 属性表：", g1.__dict__)
print("g2 属性表：", g2.__dict__)
print("不同实例：", g1 is not g2, "| 类型：", type(g1).__name__)
print("g2 是否已有 plate：", hasattr(g2, "plate"))

**预期输出**：

```text
g1 属性表： {'plate': '京A12345'}
g2 属性表： {}
不同实例： True | 类型： Gate
g2 是否已有 plate： False
```

### 11.2.5 实操步骤（上机跟随）

1. 运行代码，核对 g1 属性表： {'plate': '京A12345'}，`g1.plate = ...` 只往 `g1.__dict__` 写键。
2. 核对 g2 属性表： {}，`g2` 没有继承到 `g1` 动态挂的属性，验证两份 `__dict__` 相互独立。
3. 核对不同实例： True | 类型： Gate，`is not` 比较身份、`type().__name__` 读类名。
4. 核对 g2 是否已有 plate： False，`hasattr` 在 `g2.__dict__` 里找不到该键。
5. 把 `g1 = Gate()` 改成漏写括号的 `g1 = Gate` 重跑，会变成给类对象挂属性、属性表不再是实例属性表，反证实例化必须写括号。

### 11.2.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 漏写括号、把类对象赋值给变量后访问实例属性报 `AttributeError` | 这只是给类对象又绑了名字，并未创建实例 | 实例化必须写 `Vehicle()` |
| 类名写成小写 `class vehicle:` 被提醒不规范 | 类名约定用大驼峰 | 改为 `Vehicle`、`TrafficLight` 这类大驼峰 |
| 实例化参数个数与 `__init__` 不符报 `TypeError` | 实参会整体传给构造签名，个数必须对齐 | 对照 `__init__` 的形参考传参 |

## 11.3 知识点 11.2　`__init__` 与 `self`

### 11.3.1 核心概念

- `__init__` 是**构造方法**，实例化时自动执行，用于初始化属性；
- `self` 代表"当前这个对象"，是实例方法的**第一个参数**，由 Python 自动传入，调用时**不需要手写**；
- `self.属性 = 值` 才是"给这个对象设置属性"；不加 `self.` 的变量只是方法内的局部变量，方法结束即消失。

**原理补充（概念深化）**：`v = Vehicle(...)` 实际分两步：`__new__` 造出空实例，再以该实例为第一个实参调用 `__init__`。所谓 `self` 并不是关键字，只是约定俗成的形参名，Python 通过绑定方法机制在调用 `v.info()` 时自动把 `v` 塞进首位，所以定义必须写 `self`、调用却绝不手写——漏写会让自动传入的实例与你写的参数个数对不上，报 `takes 0 positional arguments but 1 was given`。`self.x = 值` 是往实例 `__dict__` 写键，而方法内不加 `self.` 的局部变量只是栈帧上的临时量，方法返回即随帧销毁，从 `hasattr` 和 `__dict__` 都查不到。另一个易错点是拼写：`__init__` 前后各两个下划线，写成 `__int__` 就成了普通方法，构造时不会被调用、属性也就没初始化。默认参数在定义时求值一次，放在构造签名里可让省略实参的对象也有初值。

### 11.3.2 语法要点

| 写法 | 含义 |
|---|---|
| `def __init__(self, plate, speed=0):` | 构造方法，`self` 在首位 |
| `self.plate = plate` | 把参数存为实例属性 |
| `v1 = Vehicle("京A12345")` | 调用时省略 `self` |
| `def info(self):` | 实例方法同样以 `self` 开头 |

### 11.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 定义时漏写 `self` | 调用时 `TypeError: takes 0 positional arguments but 1 was given` | 第一个参数必须是 `self` |
| 调用时手写 `self` | `TypeError` | 调用不传 `self` |
| 忘记 `self.` 前缀 | 属性"存了但读不到"、`AttributeError` | `self.speed = speed` |
| `__init__` 拼写错误 | 构造方法不执行（写成 `__int__` 等） | 前后各两个下划线 |

### 11.3.4 随堂示例：__init__ 在实例化时自动执行

**核心要点**：验证 `__init__` 的调用时机（`Vehicle(...)` 时自动执行）与 `self` 的含义（指代刚创建的那个对象）；`实例.__dict__` 可看到属性表。

In [ ]:
class Vehicle:
    def __init__(self, plate, speed=0):
        self.plate = plate          # self 指向当前实例
        self.speed = speed

v = Vehicle("京A12345", 62)
print("车牌：", v.plate, "| 车速：", v.speed)
print("属性表：", v.__dict__)

default = Vehicle("京B67890")       # 使用默认 speed=0
print("默认车速：", default.speed)

**预期输出**：

```text
车牌： 京A12345 | 车速： 62
属性表： {'plate': '京A12345', 'speed': 62}
默认车速： 0
```

**变式代码（迁移练习）**：把 demo11-2 的打印属性表加深为故意在 `__init__` 里写一个不带 `self.` 的局部变量，验证它不会进入 `__dict__`、且默认参数照常生效。

In [ ]:
# 验证 __init__ 自动执行与 self 含义：不加 self. 的变量只是局部变量
class Vehicle:
    def __init__(self, plate, speed=0):
        self.plate = plate
        self.speed = speed
        local = "临时值"                  # 无 self. 前缀，方法结束即消失
v = Vehicle("京A12345", 62)
print("属性表只含带 self 的两个：", v.__dict__)
print("local 没有成为属性：", hasattr(v, "local"))
print("默认参数生效：", Vehicle("京B67890").speed)

**预期输出**：

```text
属性表只含带 self 的两个： {'plate': '京A12345', 'speed': 62}
local 没有成为属性： False
默认参数生效： 0
```

### 11.3.5 实操步骤（上机跟随）

1. 运行代码，核对属性表只含带 self 的两个： {'plate': '京A12345', 'speed': 62}，`__init__` 在实例化时自动执行。
2. 核对 local 没有成为属性： False，方法内不带 `self.` 的 `local` 只是局部变量、方法返回即销毁。
3. 核对默认参数生效： 0，不传车速实参创建对象时取默认值 0。
4. 把不带前缀的 `local` 赋值改成 `self.local` 赋值重跑，属性表会多出 `local` 键，体会 `self.` 前缀的作用。
5. 把 `def __init__` 故意拼成 `def __int__` 重跑，构造时不再初始化属性、访问 `v.plate` 会抛 `AttributeError`。

### 11.3.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 方法里写了不带 `self.` 的赋值，之后读不到属性 | 那只是局部变量，方法结束即随栈帧销毁 | 写成 `self.speed = speed` |
| 定义带了 `self`，调用时还手写 `v.info(self)` 报 `TypeError` | `self` 由绑定方法自动传入，调用不手写 | 调用写 `v.info()`，定义才保留 `self` |
| 把 `__init__` 拼成 `__int__`，属性没初始化 | 构造方法名写错，实例化时不会被调用 | 前后各两个下划线，仔细核对拼写 |

## 11.4 知识点 11.3　实例属性与类属性

### 11.4.1 核心概念

| 类型 | 定义位置 | 归属 | 访问方式 | 用途 |
|---|---|---|---|---|
| 实例属性 | `__init__` 内 `self.x = …` | 每个对象各一份 | `obj.x` | 车牌、车速、当前相位 |
| 类属性 | 类体内、方法外 `x = …` | 全类共享一份 | `类名.x` 或 `obj.x` | 创建计数、相位顺序等常量 |

**原理补充（概念深化）**：属性读取遵循先实例 `__dict__`、再类 `__dict__`、再沿继承链的查找顺序，这正是 `obj.total` 能读到类属性的原因；但写入永远只作用于点号左边的那一个。`self.total += 1` 会被拆成读 `self.total`、加 1、写回 `self.total` 三步：读时因实例无此键而回退到类拿到 0，写时却直接在实例 `__dict__` 新建一个 `total` 键，类属性纹丝不动——于是每个对象各有一个计数器、`Vehicle.total` 恒为 0，这是本章最隐蔽的错误。要真正改类属性必须写 `类名.属性 += 1`。另一层风险是类属性若为可变对象（如列表），所有实例共享同一份引用，一处 `append` 会污染全部对象；因此每个对象各自的数据应放实例属性，类属性只放计数、常量表这类共享且通常不可变的值。误以为 `obj.total` 一定读到类属性，也会在对象已有同名实例属性时算错。

### 11.4.2 语法要点（关键区别）

- 修改类属性必须写 `类名.属性 += 1`；
- 若写 `self.属性 += 1`，会在该对象上**新建一个同名实例属性**，类属性不变——这是本章最隐蔽的错误；
- 读取时 `obj.类属性` 可用（先找实例、再找类），但修改时不建议这样写。

### 11.4.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| `self.total += 1` | 每个对象一个计数器，类属性恒为 0 | `Vehicle.total += 1` |
| 类属性用可变对象（列表） | 所有实例共享同一列表，互相污染 | 可变数据放实例属性 |
| 以为 `obj.total` 一定读到类属性 | 若该对象已有同名实例属性则读到实例的 | 用类名访问最稳 |

### 11.4.4 随堂示例：类属性共享、实例属性独立

**核心要点**：验证类属性由所有实例共享（用于计数最典型）、实例属性各自独立；通过类名或实例都能读到类属性。

In [ ]:
class Vehicle:
    total = 0                       # 类属性：所有实例共享

    def __init__(self, plate):
        self.plate = plate          # 实例属性：各自独立
        Vehicle.total += 1          # 每造一辆车，计数加一

a = Vehicle("京A12345")
b = Vehicle("京B67890")

print("a.plate：", a.plate, "| b.plate：", b.plate)
print("Vehicle.total：", Vehicle.total)
print("通过实例读取类属性：", a.total)
print("a 的属性表只有 plate：", a.__dict__)

**预期输出**：

```text
a.plate： 京A12345 | b.plate： 京B67890
Vehicle.total： 2
通过实例读取类属性： 2
a 的属性表只有 plate： {'plate': '京A12345'}
```

**变式代码（迁移练习）**：把 demo11-3 的类名正确累加故意改成 `self.total += 1`，验证实例属性会遮蔽类属性、`Vehicle.total` 恒为 0 这一最隐蔽错误。

In [ ]:
# 验证遮蔽：self.total += 1 只新建实例属性，类属性不受影响
class Vehicle:
    total = 0
    def __init__(self, plate):
        self.plate = plate
        self.total += 1                # 错误写法：实例属性遮蔽类属性
v = Vehicle("京A12345")
print("类属性仍是 0：", Vehicle.total)
print("实例读到自己的 1：", v.total)
print("实例属性表：", v.__dict__)

**预期输出**：

```text
类属性仍是 0： 0
实例读到自己的 1： 1
实例属性表： {'plate': '京A12345', 'total': 1}
```

### 11.4.5 实操步骤（上机跟随）

1. 运行代码，核对类属性仍是 0： 0，说明 `Vehicle.total` 没有被 `self.total += 1` 改动。
2. 核对实例读到自己的 1： 1，`v.total` 优先命中实例 `__dict__` 里的同名键。
3. 核对实例属性表： {'plate': '京A12345', 'total': 1}，确实多出一个 `total` 实例属性、遮蔽了类属性。
4. 把 `self.total += 1` 改回 `Vehicle.total += 1` 重跑，`Vehicle.total` 变为 1、`v.__dict__` 不再有 `total` 键。
5. 连续造两辆车再打印 `Vehicle.total`：错误写法下它恒为 0，正确写法下等于造车数，验证修改类属性必须用类名。

### 11.4.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `self.total += 1` 后 `Vehicle.total` 恒为 0 | `+=` 的写回在实例 `__dict__` 新建同名属性、遮蔽类属性 | 改类属性必须写 `Vehicle.total += 1` |
| 类属性放列表，一个对象 `append` 后所有实例都变了 | 可变类属性被所有实例共享同一份引用 | 各对象自持的数据放实例属性 |
| 以为 `obj.total` 一定读到类属性 | 若该对象已有同名实例属性，读取会先命中实例 | 用 `类名.total` 读取最稳妥 |

## 11.5 知识点 11.4　实例方法与自描述

### 11.5.1 核心概念

实例方法就是"写在类里的函数"，第一个参数固定为 `self`，内部通过 `self.属性` 读写数据、通过 `self.方法()` 调用同类其他方法。

**自描述**有两种：

| 方式 | 写法 | 触发时机 |
|---|---|---|
| 普通方法 | `def info(self): return "…"` | 显式调用 `obj.info()` |
| 魔术方法 | `def __str__(self): return "…"` | `print(obj)`、`str(obj)` 时自动调用 |

**原理补充（概念深化）**：`print(obj)` 与 `str(obj)` 并不是把对象直接变成文字，而是解释器去查该类是否有 `__str__`，有则调用它并打印返回的字符串；没有才回退到 `__repr__`，默认实现给出带内存地址的 `<模块.类 object at 0x…>` 表示，这就是不要打印对象默认 `repr` 的由来。`__str__` 必须返回 `str`，返回数字会抛 `TypeError`，所以拼接时要用 `str()` 把车速等数值显式转文本。`__repr__` 面向调试、追求无歧义，`__str__` 面向用户、追求可读，两者用途不同常被混淆。实例方法的第一个形参 `self` 由绑定机制自动传入，方法内部要调用同类其它方法必须写 `self.cycle()`；写成 `cycle()` 会在局部与全局命名空间都找不到名字而报 `NameError`。让 `info()`、`__str__` 都 `return` 字符串而非就地 `print`，结果才能被继续拼接、写日志或参与比较。

### 11.5.2 语法要点

- `__str__` 必须返回**字符串**，返回其他类型会抛 `TypeError`；
- 没有 `__str__` 时，`print(obj)` 显示 `<__main__.Vehicle object at 0x…>`；
- 方法之间可互相调用：`self.get_cycle()`，不要写 `get_cycle()`（会因缺少 `self` 报错）。

### 11.5.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| `__str__` 返回数字 | `TypeError: __str__ returned non-string` | 用 `str()` 拼接 |
| 方法内调用同类方法漏 `self.` | `NameError` | `self.方法名()` |
| 在方法里 `print` 而不 `return` | 无法参与后续拼接 | 需要组合就返回字符串 |

### 11.5.4 随堂示例：方法返回值与 __str__ 的区别

**核心要点**：验证三种输出路径：普通方法返回字符串需自己 `print`、`__str__` 决定 `print(对象)` 的显示、`str(对象)` 与 `print` 等价。

In [ ]:
class Vehicle:
    def __init__(self, plate, speed):
        self.plate = plate
        self.speed = speed

    def info(self):
        return self.plate + " " + str(self.speed) + " km/h"

    def __str__(self):
        return "Vehicle<" + self.plate + ">"

v = Vehicle("京A12345", 62)
print(v.info())            # 普通方法返回字符串：必须自己 print
print(v)                   # print 对象 → 自动调用 __str__
print(str(v))              # 显式 str() → 同样走 __str__，结果一致

**预期输出**：

```text
京A12345 62 km/h
Vehicle<京A12345>
Vehicle<京A12345>
```

**变式代码（迁移练习）**：把 demo11-4 的 `info` 与 `__str__` 并列改为让 `__str__` 内部用 `self.cycle()` 复用同类方法，验证方法互调必须带 `self.`、且 `print` 与 `str` 都触发 `__str__`。

In [ ]:
# __str__ 必须返回字符串；方法内互调带 self.，print 与 str 都触发它
class Signal:
    def cycle(self):
        return self.green * 2 + 3
    def __str__(self):
        return "绿灯 " + str(self.green) + "s / 周期 " + str(self.cycle()) + "s"
s = Signal()
s.green = 35
print("print(对象) 自动调用 __str__：", s)
print("str(对象) 结果一致：", str(s))

**预期输出**：

```text
print(对象) 自动调用 __str__： 绿灯 35s / 周期 73s
str(对象) 结果一致： 绿灯 35s / 周期 73s
```

### 11.5.5 实操步骤（上机跟随）

1. 运行代码，核对两行分别以 print(对象) 自动调用 __str__ 与 str(对象) 结果一致开头，文本完全相同。
2. 核对文本为绿灯 35s / 周期 73s，其中 73 = 35 * 2 + 3 来自 `self.cycle()`，验证方法互调生效。
3. 确认打印 `s` 没有出现带内存地址的 object at 0x 默认表示，说明自定义 `__str__` 覆盖了默认 `repr`。
4. 把 `__str__` 的返回值改成整数 `self.cycle()` 重跑，会抛 `TypeError`，验证它必须返回字符串。
5. 把 `self.cycle()` 改成 `cycle()` 重跑，会报 `NameError`，验证同类方法互调必须带 `self.`。

### 11.5.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `__str__` 返回数字时打印对象报 `TypeError` | `__str__` 必须返回 `str`，返回其他类型不合法 | 用 `str()` 把数值拼接成字符串再返回 |
| 方法里写 `cycle()` 报 `NameError` | 同类方法互调必须经 `self.` 走绑定方法 | 写成 `self.cycle()` |
| `print(对象)` 显示带内存地址的默认表示 | 没有 `__str__`，回退到默认 `__repr__` | 定义 `__str__` 返回可读文本 |

## 11.6 知识点 11.5　私有属性与受控访问

### 11.6.1 核心概念

属性名以**两个下划线开头**（`self.__green`）即成为"私有"，外部不能直接访问，只能通过公有方法读取与修改。这样可以把**校验逻辑**集中到唯一入口，防止误操作。

Python 的实现方式不是"禁止访问"，而是**名称改写**：`__green` 在类内部被改名为 `_TrafficLight__green`，因此 `hasattr(light, "__green")` 返回 `False`，但理论上仍可用改写后的名字访问。**私有是约定与防护，不是绝对隔离。**

**原理补充（概念深化）**：Python 的私有不是权限隔离，而是编译期的名称改写：类体内以双下划线开头且不以双下划线结尾的名字，会被改写成 `_类名__名字`，例如 `self.__green` 实际存成 `_TrafficLight__green`。因此用 `hasattr` 查 `__green` 返回 `False` 并非属性不存在，而是它换了名字；理论上仍可用改写名访问，故私有是约定与防护、不是绝对隔离。改写只发生在 `class` 语句内部，类外写 `light.__green = 0` 不会触发改写，只会新建一个同名普通属性，真正的私有值丝毫未变——这与讲义易错表一致。封装的价值在于把校验收进唯一入口 `set_xxx()`：`__init__` 末尾调用私有 `__check()` 可纠正非法初值，`get` 与 `set` 成对提供保证读写都经过同一段逻辑。若外部能绕过入口直接赋值，绿灯 0 秒这类误操作就无人拦截，封装形同虚设。

### 11.6.2 语法要点

| 写法 | 含义 |
|---|---|
| `self.__green = green` | 私有属性 |
| `def get_green(self): return self.__green` | 公有读接口 |
| `def set_green(self, s): 校验后赋值` | 公有写接口（**唯一修改入口**） |
| `def __check(self):` | 私有方法（内部校验） |

### 11.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 单下划线当成私有 | `_x` 只是"约定不公开"，仍可直接访问 | 需要强约束用双下划线 |
| 外部直接改私有属性 | `light.__green = 0` 会**新建**一个属性，原属性不变 | 用 `set_green()` |
| 忘记在 `__init__` 中调用校验 | 初始值非法也能创建成功 | `__init__` 末尾 `self.__check()` |
| 只提供 `set` 不提供 `get` | 外部读不到值 | 成对提供 |

---

### 11.6.4 随堂示例：私有属性只经受控方法修改

**核心要点**：验证 `__green` 这类双下划线属性在类外不可直接访问（`hasattr` 为 `False`），只能通过公有的 `get` / `set` 方法读写，从而把校验逻辑收进类内。

In [ ]:
class TrafficLight:
    def __init__(self, green):
        self.__green = green        # 私有属性

    def get_green(self):
        return self.__green

    def set_green(self, seconds):
        if 5 <= seconds <= 120:
            self.__green = seconds
            return True
        print("[拒绝] 绿灯时长必须在 5~120 秒")
        return False

light = TrafficLight(35)
print("当前绿灯：", light.get_green())
print("改为 45：", light.set_green(45), "→", light.get_green())
print("改为 200：", light.set_green(200), "→", light.get_green())
print("类外能直接访问吗：", hasattr(light, "__green"))

**预期输出**：

```text
当前绿灯： 35
改为 45： True → 45
[拒绝] 绿灯时长必须在 5~120 秒
改为 200： False → 45
类外能直接访问吗： False
```

**变式代码（迁移练习）**：把 demo11-5 的验证 `hasattr` 为假推进到看 `__dict__` 里的真实键名，验证 `__green` 被改写成 `_TrafficLight__green`、且越界值被 `set_green` 拒绝。

In [ ]:
# 验证名称改写：__green 在 __dict__ 里存为 _TrafficLight__green，故 hasattr 为 False
class TrafficLight:
    def __init__(self, green): self.__green = green
    def get_green(self): return self.__green
    def set_green(self, s): self.__green = s if 5 <= s <= 120 else self.__green
light = TrafficLight(35)
light.set_green(45)
light.set_green(200)                   # 越界被拒，私有值仍为 45
print("属性表键名：", list(light.__dict__.keys()))
print("读数/改写名：", light.get_green(), light._TrafficLight__green, "| hasattr：", hasattr(light, "__green"))

**预期输出**：

```text
属性表键名： ['_TrafficLight__green']
读数/改写名： 45 45 | hasattr： False
```

### 11.6.5 实操步骤（上机跟随）

1. 运行代码，核对属性表键名： ['_TrafficLight__green']，私有属性被改写成 `_类名__属性` 而非 `__green`。
2. 核对读数/改写名一行里 `hasattr` 为 False，`hasattr` 为假是因为名字已改写、不是属性不存在。
3. 核对 `light.set_green(200)` 被拒绝后 `get_green()` 仍为 45，验证越界值没有通过唯一入口写入。
4. 核对同行两个 45 分别是 `get_green()` 与改写名 `_TrafficLight__green` 的值，证明私有是名称改写、理论仍可访问，并非绝对隔离。
5. 在类外加一句 `light.__green = 0` 再打印属性表，会多出一个普通 `__green` 键而 `get_green()` 仍是 45，验证类外同名赋值不影响私有值。

### 11.6.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 用 `hasattr` 查 `__green` 返回 `False`，以为属性丢了 | 双下划线被改写成 `_类名__green`，不是不存在 | 知道名称改写机制，用 `get_green()` 读取 |
| 类外写 `light.__green = 0` 后绿灯值没变 | 类外不触发改写，只是新建了一个普通同名属性 | 一律用 `set_green()` 这个唯一入口修改 |
| 初始值非法也能创建成功 | `__init__` 末尾没调用校验 | 在 `__init__` 里调用私有 `__check()` 纠正初值 |

# 第二部分　代码示范

## 案例 11-1　车辆类 Vehicle

- **覆盖知识点**：11.1（类与实例）、11.2（`__init__` / `self`）、11.3（类属性 `total`）、11.4（方法、`__str__`）
- **源文件**：`case1_车辆类.py`
- **难度**：基础　**建议用时**：30 分钟

### 任务情境

仿真系统需要创建大量车辆对象，每辆车有车牌、车型、当前车速，能加速、减速、输出自身信息；同时系统要统计"累计创建了多少辆车"。

### 任务要求

1. 定义 `Vehicle` 类，`__init__` 接收车牌、车型、初始车速（默认 0）；
2. 用**类属性** `total` 统计创建数量，在 `__init__` 中累加；
3. 实现 `accelerate()` 与 `brake()`，把车速限制在 0~120；
4. 实现 `info()` 返回一行描述，并实现 `__str__` 使 `print(对象)` 可读。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 无外部输入；三个对象与其属性写在源码字面量中，不读文件、不调 `input()` |
| 目录结构 | 单文件 `case1_车辆类.py`，不依赖同章其他文件 |
| 构造参数 | `Vehicle(plate, vtype, speed=0)`：`plate`、`vtype` 为 `str`，`speed` 为 `int`，默认 0 |
| 实参取值 | `京A12345`/`小型车`/省略、`京B67890`/`大型车`/45、`冀D11223`/`新能源`/60 |
| 取值范围 | 车速被 `min(..., 120)` 与 `max(..., 0)` 夹在 0—120；加速实参 80、50，减速实参 20、100 |
| 非法值示例 | 少传 `vtype` 报 `TypeError`；写成 `self.total += 1`；加速实参传负数 |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 终端 10 行，无空行 |
| 逐行格式 | 第 1—3 行 `info()` 结果，形如 `京A12345（小型车）当前 0 km/h`；第 4—7 行形如 `v1 加速 80： 80 km/h`；第 8 行 `Vehicle<京A12345>`；第 9、10 行为累计创建数与实例读取类属性 |
| 对象字段顺序 | 每对象 3 字段：`plate`、`vtype`、`speed`；打印后 `v1` 为 0、`v2` 为 45、`v3` 为 60；加速后 `v1` 为 80 与 120，减速后 `v2` 为 25 与 0 |
| 精度 / 单位 | 车速整数、km/h；累计数带「辆」；车型用全角括号 |
| 判定方式 | 与运行结果 10 行逐字比对；重点第 5 行 `120` 与第 7 行 `0` |

### 参考代码

In [ ]:
# ---------- 第 11 章 面向对象编程（上）· 案例 11-1：车辆类 Vehicle ----------
# 对应知识点：11.1 类与实例化、11.2 __init__ 与 self、11.3 实例属性与类属性、11.4 方法与 __str__
# 功能说明：定义车辆类并创建三辆车，演示加速 / 减速的上下限约束与类属性累计创建数

class Vehicle:
    """车辆类：描述车辆属性与行驶行为"""

    total = 0                      # 类属性：所有实例共享，累计创建数

    def __init__(self, plate, vtype, speed=0):
        self.plate = plate          # 实例属性：每辆车独立
        self.vtype = vtype
        self.speed = speed
        Vehicle.total += 1          # 通过类名访问类属性

    def accelerate(self, delta):
        """加速，上限 120 km/h"""
        self.speed = min(self.speed + delta, 120)
        return self.speed

    def brake(self, delta):
        """减速，下限 0 km/h"""
        self.speed = max(self.speed - delta, 0)
        return self.speed

    def info(self):
        return self.plate + "（" + self.vtype + "）当前 " + str(self.speed) + " km/h"

    def __str__(self):
        return "Vehicle<" + self.plate + ">"


# ---------- 使用 ----------
v1 = Vehicle("京A12345", "小型车")
v2 = Vehicle("京B67890", "大型车", 45)
v3 = Vehicle("冀D11223", "新能源", 60)

print(v1.info())
print(v2.info())
print(v3.info())

print("v1 加速 80：", v1.accelerate(80), "km/h")
print("v1 继续加速 50：", v1.accelerate(50), "km/h（被限制在 120）")
print("v2 减速 20：", v2.brake(20), "km/h")
print("v2 紧急制动 100：", v2.brake(100), "km/h（被限制在 0）")

print(v1)
print("累计创建车辆：", Vehicle.total, "辆")
print("v1 的类属性视角：", v1.total)

### 运行结果

```text
京A12345（小型车）当前 0 km/h
京B67890（大型车）当前 45 km/h
冀D11223（新能源）当前 60 km/h
v1 加速 80： 80 km/h
v1 继续加速 50： 120 km/h（被限制在 120）
v2 减速 20： 25 km/h
v2 紧急制动 100： 0 km/h（被限制在 0）
Vehicle<京A12345>
累计创建车辆： 3 辆
v1 的类属性视角： 3
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `total = 0` 位于方法外 | 11.3 | 类属性，全类共享一份 |
| `def __init__(self, plate, vtype, speed=0)` | 11.2 | 构造方法，`speed` 用默认参数（第 6 章） |
| `self.plate = plate` | 11.2 | 三行实例属性，每个对象各一份 |
| `Vehicle.total += 1` | 11.3 | **必须用类名**，否则新建实例属性 |
| `min(self.speed + delta, 120)` | 11.4、6.2 | 在方法内做上限约束，所有加速都走同一逻辑 |
| `return self.speed` | 11.4 | 返回新值，便于调用方直接打印 |
| `info()` 字符串拼接 | 11.4、2.3 | `str(self.speed)` 把数字转文本 |
| `__str__` | 11.4 | `print(v1)` 自动调用 |
| `v1.total` | 11.3 | 读取时先找实例、再找类，故能读到 3 |

**改一改**：把 `Vehicle.total += 1` 改成 `self.total += 1`，重跑后打印 `Vehicle.total`，观察它为何仍是 0——这验证了"实例属性会遮蔽类属性"。

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 第 1 行报 `TypeError`，提示缺少位置参数 | 失败：构造时少传 `vtype` | 对照 `__init__` 数实参个数 |
| 第 5 行打印 `130` 而非 `120` | 失败：`accelerate` 未做上限约束 | 看是否写 `min(self.speed + delta, 120)` |
| 第 7 行打印负数 | 失败：`brake` 未做下限约束 | 看是否写 `max(self.speed - delta, 0)` |
| 第 9、10 行都是 `0` | 失败：累加写成 `self.total += 1` | 看累加用的是类名还是 `self` |
| 第 8 行出现 `<__main__.Vehicle object at 0x` | 失败：`__str__` 未定义或拼错 | 检查类体内是否有 `def __str__(self)` |

## 案例 11-2　信号灯类 TrafficLight

- **覆盖知识点**：11.5（私有属性与受控访问）、11.3（类属性 `PHASES`）、11.4（方法互调）
- **源文件**：`case2_信号灯类.py`
- **难度**：综合　**建议用时**：35 分钟

### 任务情境

路口信号灯有固定的相位顺序（红→绿→黄→红），各相位时长可配置，且时长一经设定不允许外部随意修改（防止误操作把绿灯改成 0 秒）。

### 任务要求

1. 用类属性保存相位顺序元组 `PHASES`；
2. 用 `__init__` 接收三个时长，存为**私有属性**（`__` 前缀）；
3. 提供 `get_cycle()` 计算周期总时长、`switch()` 切换相位；
4. 提供 `set_green()` 做合法性校验（必须 5~120 秒）；
5. 演示直接访问私有属性会失败。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 无外部输入；两个信号灯对象的名称与配时写在源码字面量中 |
| 目录结构 | 单文件 `case2_信号灯类.py`，不依赖同章其他文件 |
| 构造参数 | `TrafficLight(name, green=30, yellow=3, red=30)`：`name` 为 `str`，三个时长为 `int` 且可省略 |
| 实参取值 | 第一个为 `京港澳K12 东进口`/35/3/30；第二个为 `异常测试`/0，黄灯红灯取默认 |
| 取值范围 | 绿灯合法区间 5—120，越界由 `__check` 重置为 30；`PHASES` 为 3 元素元组，`index` 在 0—2 循环 |
| 非法值示例 | 构造传 `green=0`；调用 `set_green(200)`；类外写 `light.__green` 报 `AttributeError` |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 终端 12 行，无空行 |
| 逐行格式 | 第 1 行 `info()`：名称、相位、三时长与周期；第 2—5 行为当前相位与三次切换；第 6、8 行为 `set_green` 的布尔返回；第 10 行 `[警告]` 重置提示；第 12 行 `hasattr` 结果 |
| 对象字段顺序 | `info()` 顺序为 `name`、相位、`__green`、`__yellow`、`__red`、`get_cycle()`；`light` 为 35/3/30、周期 68，`light2` 为 30/3/30、周期 63 |
| 精度 / 单位 | 时长整数、单位 `s`；周期 78 等于 45 + 3 + 30；切换结果为 `红灯`、`绿灯`、`黄灯` 循环 |
| 判定方式 | 与运行结果 12 行逐字比对；重点第 9 行 `78`、第 11 行 `30s`、第 12 行 `False` |

### 参考代码

In [ ]:
# ---------- 第 11 章 面向对象编程（上）· 案例 11-2：信号灯类 TrafficLight ----------
# 对应知识点：11.5 私有属性与受控访问、11.3 类属性 PHASES、11.4 方法互调
# 功能说明：模拟信号灯相位轮转与配时；绿灯时长只能通过带校验的公有方法修改

class TrafficLight:
    """路口信号灯：相位轮转与配时"""

    PHASES = ("红灯", "绿灯", "黄灯")        # 类属性：相位顺序固定

    def __init__(self, name, green=30, yellow=3, red=30):
        self.name = name
        self.__green = green                 # 私有属性：外部不可直接改
        self.__yellow = yellow
        self.__red = red
        self.index = 0                       # 当前相位下标
        self.__check()

    # ---------- 内部校验（私有方法） ----------
    def __check(self):
        if self.__green < 5 or self.__green > 120:
            print("[警告] 绿灯时长异常：", self.__green, "秒，已重置为 30 秒")
            self.__green = 30

    # ---------- 公有方法 ----------
    def get_cycle(self):
        """信号周期总时长"""
        return self.__green + self.__yellow + self.__red

    def get_green(self):
        return self.__green

    def set_green(self, seconds):
        """设置绿灯时长，带合法性校验"""
        if seconds < 5 or seconds > 120:
            print("[拒绝] 绿灯时长必须在 5~120 秒之间")
            return False
        self.__green = seconds
        return True

    def current(self):
        return TrafficLight.PHASES[self.index]

    def switch(self):
        """切换到下一相位"""
        self.index = (self.index + 1) % len(TrafficLight.PHASES)
        return self.current()

    def info(self):
        return (self.name + "：" + self.current()
                + "，绿灯 " + str(self.__green) + "s / 黄灯 " + str(self.__yellow)
                + "s / 红灯 " + str(self.__red) + "s，周期 " + str(self.get_cycle()) + "s")


# ---------- 使用 ----------
light = TrafficLight("京港澳K12 东进口", green=35, yellow=3, red=30)
print(light.info())
print("当前相位：", light.current())
print("切换 ->", light.switch())
print("切换 ->", light.switch())
print("切换 ->", light.switch())

print("设置绿灯 45 秒：", light.set_green(45), "->", light.get_green(), "秒")
print("设置绿灯 200 秒：", light.set_green(200))

print("周期总长：", light.get_cycle(), "秒")

light2 = TrafficLight("异常测试", green=0)
print(light2.info())

# 尝试直接访问私有属性（会报 AttributeError，已注释）
# print(light.__green)
print("私有属性是否被隐藏：", hasattr(light, "__green"))

### 运行结果

```text
京港澳K12 东进口：红灯，绿灯 35s / 黄灯 3s / 红灯 30s，周期 68s
当前相位： 红灯
切换 -> 绿灯
切换 -> 黄灯
切换 -> 红灯
设置绿灯 45 秒： True -> 45 秒
[拒绝] 绿灯时长必须在 5~120 秒之间
设置绿灯 200 秒： False
周期总长： 78 秒
[警告] 绿灯时长异常： 0 秒，已重置为 30 秒
异常测试：红灯，绿灯 30s / 黄灯 3s / 红灯 30s，周期 63s
私有属性是否被隐藏： False
```

> 说明：`hasattr(light, "__green")` 返回 `False`，是因为 Python 把 `__green` **名称改写**为 `_TrafficLight__green`，而非真正"无法访问"。私有是约定与防护，不是绝对隔离。

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `PHASES = ("红灯", "绿灯", "黄灯")` | 11.3、5.3 | 类属性 + 元组：全类共享且不可改 |
| `self.__green = green` | 11.5 | 私有属性，外部不可直接赋值 |
| `self.__check()` 置于 `__init__` 末尾 | 11.5、11.4 | 构造时立即校验，非法初始值也能被纠正 |
| `__check` 双下划线 | 11.5 | 私有方法，外部不应调用 |
| `set_green()` 校验后赋值 | 11.5 | **唯一修改入口**，保证校验一定执行 |
| `(self.index + 1) % len(...)` | 11.4、2.4 | 取余实现相位循环，避免写四个 `if` |
| `self.current()` | 11.4 | 方法内调用同类方法，必须带 `self.` |
| `info()` 调 `self.get_cycle()` | 11.4 | 复用公有方法，不重复写求和 |
| `hasattr(light, "__green")` | 11.5 | 验证名称改写：返回 `False` |

**改一改**：把 `set_green` 里的校验删掉，再调用 `light.set_green(0)`，观察周期变成 33 秒（0 + 3 + 30）、"绿灯 0s"照样被写入——说明"没有唯一入口，封装就形同虚设"；再取消 `# print(light.__green)` 的注释，记录 `AttributeError` 的完整信息。

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 第 9 行周期仍是 `68` | 失败：`set_green(45)` 未真正写入 | 看是否执行 `self.__green = seconds` |
| 第 7 行没有 `[拒绝]`，第 8 行返回 `True` | 失败：`set_green` 缺 5—120 校验 | 核对 `if seconds < 5 or seconds > 120` |
| 第 10、11 行缺失 | 失败：`__init__` 末尾漏调 `self.__check()` | 查看构造方法最后一行 |
| 第 12 行打印 `True` | 失败：属性只有一个下划线 | 看定义处是否为 `self.__green` |
| 第 3—5 行不循环或报 `IndexError` | 失败：`switch` 漏了取余 | 看是否有 `% len(TrafficLight.PHASES)` |

---

# 第三部分　布置练习

## 练习 11-1　停车场类 ParkingLot

- **对应知识点**：11.1 — 11.5（含私有属性、`__str__` 可选）
- **提交文件**：`practice1_停车场类.py`
- **难度**：综合　**建议用时**：35 分钟

### 任务情境

停车场余位统计程序：`ParkingLot` 记住名称、总车位与不对外公开的已停车辆列表。总车位 4 时第 5 辆必须拒绝，已停在场内的 `京A12345` 再来要提示重复；驶出后余位与使用率同步变成 1 和 75.0%，总车位为 0 时不得因除零崩溃。

### 题目

定义 `ParkingLot` 类，描述一个智慧停车场：有名称、总车位数，能记录已停车辆车牌，支持驶入、驶出、查询剩余车位与使用率。

### 要求

1. `__init__` 初始化名称、总车位数、私有车辆列表；
2. `park(plate)`：车位已满时拒绝并返回 `False`，车牌已存在时提示重复；
3. `leave(plate)`：车辆不在场时提示；
4. `free()` 返回剩余车位，`usage()` 返回使用率（保留 1 位小数的百分比）；
5. 演示满位与重复入场两种情况。

### 提示

- 私有列表要用 `self.__cars = []`；判断车牌是否已存在可用 `in`。
- `usage()` 注意分母为 0 的情况（总车位为 0 时直接返回 0）。
- 三个判断的顺序有讲究：**先判重复、再判满位**。若先判满位，满场时"重复入场"会被误报成"车位已满"。

### 自检要点（不含答案）

- [ ] 总车位 4 时，第 5 辆车被拒绝，第 1 辆车重复入场被提示"已在场内"；
- [ ] 驶出后剩余车位与使用率同步变化；
- [ ] 总车位为 0 时 `usage()` 返回 0，不抛 `ZeroDivisionError`；
- [ ] 外部无法直接修改 `__cars`，只能通过 `park` / `leave`。

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 直接运行 `practice1_停车场类.py`（总车位 4） | 10 行，第 5 行 `[拒绝] B 区智慧停车场 车位已满，无法驶入`，第 6 行 `[提示] 京A12345 已在场内` | 与答案 11-1 运行结果 10 行逐字比对 |
| T2 | 边界 | 把总车位由 4 改为 0 | 仍 10 行；6 次入场全为 `[拒绝]`，两行 `info()` 末段均 `使用率 0%` | 数行数为 10，核对末段与是否抛 `ZeroDivisionError` |
| T3 | 异常 | 把 `park` 两个判断顺序颠倒，先判满位 | 仍 10 行，第 6 行变 `[拒绝] B 区智慧停车场 车位已满，无法驶入` | 只读第 6 行，出现 `[拒绝]` 即判失败 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 类与私有列表 | 阅读源码并打印属性表 | `__init__` 三属性齐全，列表为双下划线私有 | 2 |
| 驶入两条分支 | 读第 5、6 行 | 满位与重复各提示一次，顺序正确 | 3 |
| 驶出与提示 | 读第 8、9 行 | 驶出后剩余车位 1，不在场有 `[提示]` | 2 |
| 使用率计算 | 读第 7、10 行 | 100.0% 与 75.0%，均 1 位小数 | 2 |
| 零车位不崩 | 总车位改 0 后运行 | 返回 0，无 `ZeroDivisionError` | 1 |
| 合计 | — | — | 10 |

## 练习 11-2　路口类 Intersection

- **对应知识点**：11.2（`__init__`）、11.3（类属性计数）、11.4（方法）、11.5（私有计数器）
- **提交文件**：`practice2_路口类.py`
- **难度**：综合　**建议用时**：30 分钟

### 任务情境

路口设备改造后，你要用 `Intersection` 对象替代散落的字典台账：名称与车道数之外，累计通过车辆数与车速总和放在私有计数器里，只能由 `pass_car()` 更新。`京港澳 K12` 通过 5 辆车后平均车速应为 87.0；传入 400 必须被拒绝且不计数。

### 题目

定义 `Intersection` 类描述一个路口：有名称、车道数，能累计统计通过车辆数与车速总和，输出平均车速与通行能力。

### 要求

1. `__init__` 初始化名称、车道数、私有计数器 `__count` 与 `__speed_sum`；
2. `pass_car(speed)`：累计车辆数与车速总和（车速非法时拒绝）；
3. `avg_speed()` 返回平均车速（无车时返回 0）；
4. `capacity()` 返回理论通行能力（车道数 × 2200）；
5. 用类属性统计创建了多少个路口对象。

### 提示

- 车速合法性判断复用第 9 章思路：`if speed < 0 or speed > 300: 拒绝`。
- 无车时平均车速的分母为 0，必须先判断 `__count`。
- 未通车路口的 `info()` 应能正常输出（平均车速显示 0），不要因除零而崩溃。

### 自检要点（不含答案）

- [ ] 5 辆合法车 + 1 次非法车速后，`__count` 为 5、`__speed_sum` 为 435；
- [ ] 平均车速 87.0，通行能力 6600（3 车道）；
- [ ] 新建路口的平均车速为 0，不报错；
- [ ] `Intersection.total` 为 2。

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 直接运行 `practice2_路口类.py`，5 辆合法车后传 400 | 5 行，首行 `[拒绝] 车速异常： 400`，第 2 行 `已通过 5 辆，平均车速 87.0 km/h` | 与答案 11-2 运行结果 5 行逐字比对 |
| T2 | 边界 | 把 `a.pass_car(400)` 改为 `a.pass_car(300)` | 变 4 行（无拒绝行），第 1 行 `已通过 6 辆，平均车速 122.5 km/h` | 数行数为 4，核对 122.5 |
| T3 | 异常 | 删掉 `if self.__count == 0: return 0` 两行 | 打印未通车路口时报 `ZeroDivisionError: division by zero` | 读报错末行，出现 `ZeroDivisionError` 即失败 |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 私有计数器 | 打印属性表 | 出现 `_Intersection__count` 与 `_Intersection__speed_sum` | 2 |
| 非法车速拒绝 | 读第 1 行与计数 | 有 `[拒绝] 车速异常： 400`，计数仍为 5 | 3 |
| 平均值与通行能力 | 读第 2 行 | 87.0 与 6600 同时正确 | 2 |
| 零车状态可用 | 读第 3 行 | 平均车速 0、通行能力 4400，不报错 | 2 |
| 类属性计数 | 读第 5 行 | `Intersection.total` 为 2 | 1 |
| 合计 | — | — | 10 |

---

# 第四部分　习题答案

## 答案 11-1　停车场类 ParkingLot

> 对应 **练习 11-1**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 11 章 面向对象编程（上）· 练习 11-1 参考答案：停车场类 ParkingLot ----------
# 对应知识点：11.1—11.5（构造方法、私有列表、方法约束、使用率计算）
# 功能说明：实现入场、出场、余位查询与使用率统计，处理满位与重复入场两种情况

class ParkingLot:
    """智慧停车场"""

    def __init__(self, name, capacity):
        self.name = name
        self.capacity = capacity
        self.__cars = []                 # 私有：已停车辆

    def park(self, plate):
        if plate in self.__cars:
            print("[提示]", plate, "已在场内")
            return False
        if len(self.__cars) >= self.capacity:
            print("[拒绝]", self.name, "车位已满，无法驶入")
            return False
        self.__cars.append(plate)
        print("[驶入]", plate, "剩余车位", self.free())
        return True

    def leave(self, plate):
        if plate not in self.__cars:
            print("[提示]", plate, "不在本停车场")
            return False
        self.__cars.remove(plate)
        print("[驶出]", plate, "剩余车位", self.free())
        return True

    def free(self):
        return self.capacity - len(self.__cars)

    def usage(self):
        if self.capacity == 0:
            return 0
        return round(len(self.__cars) / self.capacity * 100, 1)

    def info(self):
        return (self.name + "：车位 " + str(self.capacity)
                + "，已停 " + str(len(self.__cars))
                + "，使用率 " + str(self.usage()) + "%")


lot = ParkingLot("B 区智慧停车场", 4)
lot.park("京A12345")
lot.park("京B67890")
lot.park("津C24680")
lot.park("冀D11223")
lot.park("京E55667")          # 满位
lot.park("京A12345")          # 重复入场
print(lot.info())

lot.leave("京B67890")
lot.leave("京X99999")         # 不在场
print(lot.info())

### 运行结果

```text
[驶入] 京A12345 剩余车位 3
[驶入] 京B67890 剩余车位 2
[驶入] 津C24680 剩余车位 1
[驶入] 冀D11223 剩余车位 0
[拒绝] B 区智慧停车场 车位已满，无法驶入
[提示] 京A12345 已在场内
B 区智慧停车场：车位 4，已停 4，使用率 100.0%
[驶出] 京B67890 剩余车位 1
[提示] 京X99999 不在本停车场
B 区智慧停车场：车位 4，已停 3，使用率 75.0%
```

### 解析

1. **判断顺序决定提示正确性**：满场时先判"是否已在场内"，才能对重复入场给出"已在场内"而非"车位已满"。先判满位会让提示误导车主。
2. **私有列表的必要性**：`self.__cars` 只能由 `park` / `leave` 修改，保证"已停数量"与"剩余车位"始终自洽；若外部可直接 `lot.__cars.clear()`，所有统计都会失真。
3. **返回值约定**：成功 `True`、失败 `False`，调用方可据此继续处理（如引导车主去另一个停车场），这比只打印提示更利于集成。
4. **`usage()` 的零分母保护**：总车位 0 的停车场无业务意义，但代码必须不崩，与第 9 章的"先判断再计算"一脉相承。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 满场时提示"车位已满"而非"已在场内" | 判断顺序颠倒 | 先判重复 |
| `AttributeError: __cars` | 用了 `self.cars` 定义、`self.__cars` 访问 | 名称统一 |
| 使用率 100.0 而非 75.0 | `usage` 用的 `capacity` 是剩余车位 | 分母用总车位 |
| 驶出不在场的车不提示 | `remove` 直接抛 `ValueError` | 先用 `in` 判断 |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 类结构与私有列表 | 3 | `__init__` 三属性，列表私有 |
| 驶入判断 | 4 | 重复与满位两种情况提示正确 |
| 驶出与统计 | 3 | 剩余车位、使用率同步 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 10 行输出与期望逐字一致 | 正常路径；第 4 次入场后余位为 0 |
| T2 | 总车位改 0 后仍 10 行，入场全被拒绝，info 末段 `使用率 0%` | `usage()` 先判分母为 0 |
| T3 | 判断颠倒后第 6 行变成满位 `[拒绝]` | 先判重复才能正确提示 |

## 答案 11-2　路口类 Intersection

> 对应 **练习 11-2**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 11 章 面向对象编程（上）· 练习 11-2 参考答案：路口类 Intersection ----------
# 对应知识点：11.2 __init__、11.3 类属性计数、11.4 方法互调、11.5 私有累计量
# 功能说明：统计路口通过车辆数与平均车速，非法车速拒绝计入，并统计创建的路口对象数

class Intersection:
    """路口：通行统计"""

    total = 0                          # 类属性：路口对象总数

    def __init__(self, name, lanes):
        self.name = name
        self.lanes = lanes
        self.__count = 0
        self.__speed_sum = 0
        Intersection.total += 1

    def pass_car(self, speed):
        if speed < 0 or speed > 300:
            print("[拒绝] 车速异常：", speed)
            return False
        self.__count += 1
        self.__speed_sum += speed
        return True

    def avg_speed(self):
        if self.__count == 0:
            return 0
        return round(self.__speed_sum / self.__count, 1)

    def capacity(self):
        return self.lanes * 2200

    def info(self):
        return (self.name + "：" + str(self.lanes) + " 车道，已通过 "
                + str(self.__count) + " 辆，平均车速 " + str(self.avg_speed())
                + " km/h，通行能力 " + str(self.capacity()) + " 辆/h")


a = Intersection("京港澳 K12", 3)
b = Intersection("京开 K8", 2)

for s in [62, 118, 45, 132, 78]:
    a.pass_car(s)
a.pass_car(400)                        # 非法车速

print(a.info())
print(b.info(), "（尚未通车）")
print("平均车速：", a.avg_speed(), "km/h")
print("创建路口对象：", Intersection.total, "个")

### 运行结果

```text
[拒绝] 车速异常： 400
京港澳 K12：3 车道，已通过 5 辆，平均车速 87.0 km/h，通行能力 6600 辆/h
京开 K8：2 车道，已通过 0 辆，平均车速 0 km/h，通行能力 4400 辆/h（尚未通车）
平均车速： 87.0 km/h
创建路口对象： 2 个
```

### 解析

1. **两个私有计数器**：`__count` 与 `__speed_sum` 必须**一起**更新，否则平均值失真。把它们设为私有，避免外部只改其中一个。
2. **非法车速直接返回**：`pass_car(400)` 不增加计数，因此 5 辆车、总和 435、平均 87.0——若非法值被计入，平均会变成 145.8，明显失真。
3. **类属性计数**：`Intersection.total += 1` 在 `__init__` 中执行，每创建一个对象加 1，与案例 11-1 的 `Vehicle.total` 同一模式。
4. **未通车路口**：`b` 的 `__count` 为 0，`avg_speed()` 走"返回 0"分支，`info()` 仍能完整输出，体现"零状态"也要可用。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 平均车速 145.8 | 非法车速也被计入 | 校验失败直接 `return False` |
| `ZeroDivisionError` | `avg_speed` 未判 `__count` | 先判零 |
| `Intersection.total` 恒为 0 | 写成了 `self.total += 1` | 用类名 |
| 通行能力按剩余车道算 | 公式写错 | `lanes * 2200` |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 构造函数与私有计数器 | 3 | 两个计数器初值 0 |
| 非法车速拒绝 | 3 | 不计数、有提示 |
| 平均值与通行能力 | 3 | 87.0 / 6600 / 0 |
| 类属性计数 | 1 | `total` 为 2 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 5 行输出与期望一致，首行 `[拒绝] 车速异常： 400` | 5 辆合法车合计 435，平均 87.0 |
| T2 | 改成 300 后无拒绝行，4 行，计数 6、平均 122.5 | 只有大于 300 才拒绝 |
| T3 | 删零车保护后抛 `ZeroDivisionError` | `avg_speed` 必须先判零 |

---

## 本章小结

- **类与对象**：类名大驼峰；实例化必须写括号；一个类可造多个对象，属性各自独立。
- **`__init__` 与 `self`**：构造时自动执行；`self` 自动传入、调用不写；属性必须带 `self.`。
- **两类属性**：实例属性各一份；类属性全类共享，**修改必须用类名**。
- **方法**：第一个参数 `self`，内部互调用 `self.方法()`；`__str__` 决定 `print(对象)` 的输出。
- **封装**：双下划线私有 + 公有 `get` / `set`；私有是**名称改写**而非绝对隔离；修改入口唯一，校验才可靠。

## 自检清单

- [ ] 能写出完整的类：`class` + `__init__` + 若干方法 + `self`；
- [ ] 能区分类属性与实例属性，并说明为什么 `self.total += 1` 是错的；
- [ ] 知道 `__init__` 拼写错误会导致构造方法不执行；
- [ ] 能实现 `__str__` 并解释它何时被调用；
- [ ] 会用私有属性 + `set_xxx()` 做受控访问；
- [ ] 知道 `hasattr(obj, "__x")` 为 `False` 的真实原因。

## 前后章节衔接

- 本章的类将在**第 12 章**扩展为继承体系：`Car` / `Truck` / `Bus` 继承 `Vehicle`，子类只需重写差异部分。
- 本章的私有属性与 `set_green()` 校验，将在**第 12 章**升级为"封装"的完整讨论。
- 本章的类属性 `total`，在**终结性考核**中会以 `SignalPlan.GREEN_MAP` 这类"类级常量表"形式出现。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：模块与程序包](../10模块与程序包/10_模块与程序包.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：面向对象编程下 ▶](../12面向对象编程下/12_面向对象编程下.ipynb)